<a href="https://colab.research.google.com/github/YasserJEP/TDA-BifurcacionesHopf/blob/main/Notebooks/5_Estimaci%C3%B3n_Par%C3%A1metros_InfoMutua_FNN_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hopf Normal

Este notebook analiza la serie temporal del sistema Hopf normal en una cerca al valor crítico de bifurcación (μ=0.001) para optimizar el embedding de Takens. Calcula información mutua para determinar el retardo óptimo (τ) y False Nearest Neighbors (FNN) para la dimensión óptima (m). Incluye visualización de la serie temporal y proyecciones del embedding.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from sklearn.neighbors import NearestNeighbors
from gtda.time_series import SingleTakensEmbedding
import pandas as pd
import os

# ======================================================
# 1. SISTEMA NORMAL DE HOPF
# ======================================================

def hopf_normal(t, X, mu, omega):
    """Ecuaciones diferenciales del sistema Hopf normal."""
    x, y = X
    return [
        mu * x - omega * y - x * (x**2 + y**2),
        omega * x + mu * y - y * (x**2 + y**2)
    ]

# ======================================================
# 2. GENERAR SERIE TEMPORAL x(t)
# ======================================================

def generate_hopf_ts(mu, dt=0.01, T=200):
    """Genera la serie temporal x(t) del sistema Hopf normal."""
    t_span = (0, T)
    t_eval = np.arange(0, T, dt)
    x0 = [0, 1.01]  # Condición inicial

    sol = solve_ivp(
        hopf_normal,
        t_span,
        x0,
        t_eval=t_eval,
        args=(mu, 6),
        rtol=1e-11,
        atol=1e-12
    )
    return sol.y[0]  # Retorna x(t)

# ======================================================
# 3. CONFIGURACIÓN DE PARÁMETROS
# ======================================================

mu = 0.001  # Parámetro de bifurcación (cerca de la bifurcación de Hopf)
dt = 0.01   # Paso de tiempo
T = 100     # Tiempo total de simulación

# Parámetros de embedding
tau = 27    # Retardo temporal (se recalculará con información mutua)
m = 2       # Dimensión de embedding

# ======================================================
# 4. GENERAR Y GRAFICAR LA SERIE TEMPORAL
# ======================================================

ts = generate_hopf_ts(mu, dt, T)
time = np.arange(0, T, dt)

fig = plt.figure(figsize=(15, 10))

# Subplot 1: Serie temporal completa
ax1 = plt.subplot(2, 2, 1)
ax1.plot(time, ts, 'b-', linewidth=0.8, alpha=0.8)
ax1.set_xlabel('Tiempo (t)', fontsize=12)
ax1.set_ylabel('x(t)', fontsize=12)
ax1.set_title(f'Serie Temporal del Sistema Hopf Normal (μ={mu})', fontsize=14)
ax1.margins(x=0.0)
ax1.grid(True, alpha=0.3)

# Subplot 2: Zoom
ax2 = plt.subplot(2, 2, 2)
zoom_start, zoom_end = 0, T
zoom_mask = (time >= zoom_start) & (time <= zoom_end)
ax2.plot(time[zoom_mask], ts[zoom_mask], 'r-', linewidth=0.8)
ax2.set_xlabel('Tiempo (t)', fontsize=12)
ax2.set_ylabel('x(t)', fontsize=12)
ax2.set_title(f'Zoom: t ∈ [{zoom_start}, {zoom_end}]', fontsize=14)
ax2.margins(x=0.0)
ax2.grid(True, alpha=0.3)

# ======================================================
# 5. CREAR EMBEDDING
# ======================================================

embedder = SingleTakensEmbedding(
    parameters_type="fixed",
    n_jobs=1,
    time_delay=tau,
    dimension=m,
    stride=2
)

X = embedder.fit_transform(ts[zoom_mask].reshape(-1, 1))
N = X.shape[0]

print(f"Dimensiones del embedding: {X.shape}")
print(f"Parámetros: τ={tau}, m={m}")
print(f"Número de puntos en el espacio embebido: {N}")

# ======================================================
# 6. PROYECCIONES 2D DEL EMBEDDING
# ======================================================

fig2, axes = plt.subplots(1, 2, figsize=(12, 5))

# Proyección XY
scatter1 = axes[0].scatter(X[:, 0], X[:, 1], c=np.arange(N), cmap='viridis', s=5, alpha=0.7)
axes[0].set_xlabel(f'x(t)')
axes[0].set_ylabel(f'x(t+{tau})')
axes[0].set_title('Proyección XY (Embedding 2D)')
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=axes[0], label='Tiempo')

# Proyección XZ (si m>=3)
if m >= 3:
    axes[1].scatter(X[:, 0], X[:, 2], c=np.arange(N), cmap='viridis', s=5, alpha=0.7)
    axes[1].set_xlabel(f'x(t)')
    axes[1].set_ylabel(f'x(t+{2*tau})')
    axes[1].set_title('Proyección XZ')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ======================================================
# 7. INFORMACIÓN MUTUA (CÁLCULO DE τ ÓPTIMO)
# ======================================================

max_lag = 100

def mutual_information_estimate(series, max_lag=max_lag):
    """
    Calcula la información mutua para diferentes lags.
    El primer mínimo indica el τ óptimo para embedding.
    """
    n_bins = 20
    mi_vals = []

    for lag in range(1, max_lag + 1):
        shifted = series[lag:]
        original = series[:-lag]
        hist, _, _ = np.histogram2d(original, shifted, bins=n_bins)
        hist = hist / np.sum(hist)
        mi = 0
        for i in range(n_bins):
            for j in range(n_bins):
                if hist[i, j] > 0:
                    p_x = np.sum(hist[i, :])
                    p_y = np.sum(hist[:, j])
                    mi += hist[i, j] * np.log(hist[i, j] / (p_x * p_y))
        mi_vals.append(mi)
    return np.array(mi_vals)

# Calcular información mutua
mi_vals = mutual_information_estimate(ts, max_lag=max_lag)
first_min = np.argmin(mi_vals) + 1

# Graficar información mutua
plt.figure(figsize=(8, 4))
plt.plot(range(1, max_lag + 1), mi_vals, 'b-', linewidth=1.5)
plt.axvline(x=first_min, color='g', linestyle='--', alpha=0.7, label=f'Primer mínimo: {first_min}')
plt.xlabel('Lag (τ)', fontsize=13)
plt.ylabel('Información Mutua', fontsize=13)
plt.title('Información Mutua vs Lag (Desfase temporal)', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"figure_hopf_mutual_information.pdf", bbox_inches="tight", dpi=600)
plt.show()

# Actualizar tau con el valor óptimo
tau_optimo = first_min
print(f"\nτ óptimo sugerido por información mutua: {tau_optimo}")

# ======================================================
# 8. FNN (FALSE NEAREST NEIGHBORS) - CÁLCULO DE m ÓPTIMO
# ======================================================

max_dim = 12

def compute_FNN_curve(ts, tau, max_dim=max_dim, Rtol=15, Atol=2):
    """
    Calcula el porcentaje de falsos vecinos cercanos (FNN).
    El mínimo indica la dimensión óptima m para embedding.
    """
    N = len(ts)
    perc_FNN = []

    for m in range(1, max_dim + 1):
        M = N - (m * tau)
        if M <= 0:
            break

        # Construir vectores en dimensión m
        Y_m = np.zeros((M, m))
        for i in range(m):
            Y_m[:, i] = ts[i * tau : i * tau + M]

        # Construir vectores en dimensión m+1
        Y_m1 = np.zeros((M, m + 1))
        for i in range(m + 1):
            Y_m1[:, i] = ts[i * tau : i * tau + M]

        # Encontrar vecinos más cercanos
        nbrs = NearestNeighbors(n_neighbors=2).fit(Y_m)
        distances, indices = nbrs.kneighbors(Y_m)

        R_m = distances[:, 1]
        nn_index = indices[:, 1]

        # Calcular distancias en dimensión m+1
        dist_m1 = np.linalg.norm(Y_m1 - Y_m1[nn_index], axis=1)

        # Condiciones para identificar falsos vecinos
        condition1 = np.abs(dist_m1 - R_m) / R_m > Rtol
        condition2 = dist_m1 > Atol

        FNN = np.sum(condition1 | condition2)
        perc_FNN.append(100 * FNN / M)

    return perc_FNN

# Calcular FNN
perc_manual = compute_FNN_curve(ts, tau_optimo, max_dim=max_dim)

# Graficar FNN
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(perc_manual) + 1), perc_manual, marker='o', color='b', linewidth=1.5)
plt.xlabel("Dimensión m", fontsize=13)
plt.ylabel("Porcentaje FNN", fontsize=13)
plt.title("False Nearest Neighbors (FNN)", fontsize=13)
plt.xlim(0, max_dim)
plt.xticks(np.arange(0, max_dim + 1, 1))
plt.yticks(np.arange(0, 101, 10))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"figure_hopf_FNN.pdf", bbox_inches="tight", dpi=600)
plt.show()

# Encontrar dimensión óptima (primer valor donde FNN < 10%)
m_optimo = 2
for i, val in enumerate(perc_manual):
    if val < 10:
        m_optimo = i + 1
        break

print(f"\nDimensión m óptima sugerida por FNN: {m_optimo}")

# ======================================================
# 9. RESUMEN DEL ANÁLISIS
# ======================================================

print("\n" + "="*50)
print("RESUMEN DEL ANÁLISIS")
print("="*50)
print(f"Longitud de la serie: {len(ts)} puntos")
print(f"Tiempo total: {T} unidades")
print(f"Resolución temporal: {dt}")
print(f"Parámetro μ: {mu}")
print(f"\nParámetros de embedding sugeridos:")
print(f"  - Retardo óptimo (τ): {tau_optimo} (primer mínimo información mutua)")
print(f"  - Dimensión óptima (m): {m_optimo} (FNN < 10%)")
print(f"\nParámetros usados originalmente:")
print(f"  - Dimensión (m): {m}")
print(f"  - Retardo (τ): {tau}")

# Lorenz

Este notebook analiza la serie temporal del sistema Lorenz cercano a (ρ=24.74) para optimizar el embedding de Takens. Calcula autocorrelación e información mutua para determinar el retardo óptimo (τ), y False Nearest Neighbors (FNN) para la dimensión óptima (m). Incluye visualización de la serie temporal y proyecciones del embedding.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from sklearn.neighbors import NearestNeighbors
from gtda.time_series import SingleTakensEmbedding

# ======================================================
# 1. SISTEMA DE LORENZ
# ======================================================

def lorenz(t, X, sigma, beta, rho):
    """Ecuaciones diferenciales del sistema Lorenz."""
    x, y, z = X
    return [
        sigma * (y - x),
        rho * x - y - x * z,
        x * y - beta * z
    ]

# ======================================================
# 2. GENERAR SERIE TEMPORAL x(t)
# ======================================================

def generate_lorenz_ts(rho, dt=0.01, T=200):
    """Genera la serie temporal x(t) del sistema Lorenz."""
    t_span = (0, T)
    t_eval = np.arange(0, T, dt)
    x0 = [6.74, 6.74, 22.74]  # Condiciones iniciales

    sol = solve_ivp(
        lorenz,
        t_span,
        x0,
        t_eval=t_eval,
        args=(10.0, 8/3, rho),
        rtol=1e-11,
        atol=1e-12
    )
    return sol.y[0]  # Retorna x(t)

# ======================================================
# 3. CONFIGURACIÓN DE PARÁMETROS
# ======================================================

rho = 24.73      # Valor caótico clásico
dt = 0.01        # Paso de tiempo
T = 100          # Tiempo total de simulación

# Parámetros de embedding (se optimizarán después)
tau = 16         # Retardo temporal inicial
m = 2            # Dimensión de embedding inicial

# ======================================================
# 4. GENERAR Y GRAFICAR LA SERIE TEMPORAL
# ======================================================

ts = generate_lorenz_ts(rho, dt, T)
time = np.arange(0, T, dt)

fig = plt.figure(figsize=(15, 10))

# Subplot 1: Serie temporal completa
ax1 = plt.subplot(2, 2, 1)
ax1.plot(time, ts, 'b-', linewidth=0.8, alpha=0.8)
ax1.set_xlabel('Tiempo (t)', fontsize=12)
ax1.set_ylabel('x(t)', fontsize=12)
ax1.set_title(f'Serie Temporal del Sistema de Lorenz (ρ={rho})', fontsize=14)
ax1.margins(x=0.0)
ax1.grid(True, alpha=0.3)

# Subplot 2: Zoom de la serie temporal
ax2 = plt.subplot(2, 2, 2)
zoom_start, zoom_end = 0, T
zoom_mask = (time >= zoom_start) & (time <= zoom_end)
ax2.plot(time[zoom_mask], ts[zoom_mask], 'r-', linewidth=0.8)
ax2.set_xlabel('Tiempo (t)', fontsize=12)
ax2.set_ylabel('x(t)', fontsize=12)
ax2.set_title(f'Zoom: t ∈ [{zoom_start}, {zoom_end}]', fontsize=14)
ax2.margins(x=0.0)
ax2.grid(True, alpha=0.3)

# ======================================================
# 5. CREAR EMBEDDING DE TAKENS
# ======================================================

embedder = SingleTakensEmbedding(
    parameters_type="fixed",
    n_jobs=1,
    time_delay=tau,
    dimension=m,
    stride=2
)

X = embedder.fit_transform(ts[zoom_mask].reshape(-1, 1))
N = X.shape[0]

print(f"Dimensiones del embedding: {X.shape}")
print(f"Parámetros: τ={tau}, m={m}")
print(f"Número de puntos en el espacio embebido: {N}")

# ======================================================
# 6. PROYECCIONES 2D DEL EMBEDDING
# ======================================================

fig2, axes = plt.subplots(1, 2, figsize=(12, 5))

# Proyección XY
scatter1 = axes[0].scatter(X[:, 0], X[:, 1], c=np.arange(N), cmap='viridis', s=5, alpha=0.7)
axes[0].set_xlabel(f'x(t)')
axes[0].set_ylabel(f'x(t+{tau})')
axes[0].set_title('Proyección XY')
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=axes[0], label='Tiempo')

# Proyección XZ (si m>=3)
if m >= 3:
    scatter2 = axes[1].scatter(X[:, 0], X[:, 2], c=np.arange(N), cmap='viridis', s=5, alpha=0.7)
    axes[1].set_xlabel(f'x(t)')
    axes[1].set_ylabel(f'x(t+{2*tau})')
    axes[1].set_title('Proyección XZ')
    axes[1].grid(True, alpha=0.3)
    plt.colorbar(scatter2, ax=axes[1], label='Tiempo')

plt.tight_layout()
plt.show()

# ======================================================
# 7. ANÁLISIS DE AUTOCORRELACIÓN
# ======================================================

fig3, axes = plt.subplots(1, 2, figsize=(12, 5))

# Autocorrelación
from scipy.signal import correlate
autocorr = correlate(ts - np.mean(ts), ts - np.mean(ts), mode='full')
autocorr = autocorr[len(autocorr)//2:]
autocorr = autocorr / autocorr[0]

# Encontrar primer cruce por cero
zero_crossings = np.where(np.diff(np.sign(autocorr)))[0]
first_zero_crossing = zero_crossings[0] if len(zero_crossings) > 0 else len(autocorr)-1

axes[0].plot(autocorr[:200], 'b-', linewidth=1.5)
axes[0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[0].axvline(x=first_zero_crossing, color='g', linestyle='--',
                alpha=0.7, label=f'Primer cruce: {first_zero_crossing}')
axes[0].set_xlabel('Lag (k)')
axes[0].set_ylabel('Autocorrelación')
axes[0].set_title('Autocorrelación de x(t)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ======================================================
# 8. INFORMACIÓN MUTUA
# ======================================================
max_lag = 80
def mutual_information_estimate(series, max_lag=max_lag):
    """Calcula la información mutua para diferentes lags."""
    n_bins = 20
    mi_vals = []

    for lag in range(1, max_lag + 1):
        shifted = series[lag:]
        original = series[:-lag]

        hist, _, _ = np.histogram2d(original, shifted, bins=n_bins)
        hist = hist / np.sum(hist)

        mi = 0
        for i in range(n_bins):
            for j in range(n_bins):
                if hist[i, j] > 0:
                    p_x = np.sum(hist[i, :])
                    p_y = np.sum(hist[:, j])
                    mi += hist[i, j] * np.log(hist[i, j] / (p_x * p_y))
        mi_vals.append(mi)

    return np.array(mi_vals)

mi_vals = mutual_information_estimate(ts, max_lag=max_lag)
first_min = np.argmin(mi_vals) + 1

axes[1].plot(range(1, 51), mi_vals, 'b-', linewidth=1.5)
axes[1].axvline(x=first_min, color='g', linestyle='--',
                alpha=0.7, label=f'Primer mínimo: {first_min}')
axes[1].set_xlabel('Lag (τ)')
axes[1].set_ylabel('Información Mutua')
axes[1].set_title('Información Mutua vs Lag')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Actualizar tau con el valor óptimo
tau_optimo = first_min
print(f"\nτ óptimo sugerido por información mutua: {tau_optimo}")

# ======================================================
# 9. FNN (FALSE NEAREST NEIGHBORS)
# ======================================================

max_dim = 12

def compute_FNN_curve(ts, tau, max_dim=max_dim, Rtol=15, Atol=2):
    """Calcula el porcentaje de falsos vecinos cercanos (FNN)."""
    N = len(ts)
    perc_FNN = []

    for m in range(1, max_dim + 1):
        M = N - (m * tau)
        if M <= 0:
            break

        # Vectores en dimensión m
        Y_m = np.zeros((M, m))
        for i in range(m):
            Y_m[:, i] = ts[i * tau : i * tau + M]

        # Vectores en dimensión m+1
        Y_m1 = np.zeros((M, m + 1))
        for i in range(m + 1):
            Y_m1[:, i] = ts[i * tau : i * tau + M]

        # Vecinos más cercanos
        nbrs = NearestNeighbors(n_neighbors=2).fit(Y_m)
        distances, indices = nbrs.kneighbors(Y_m)

        R_m = distances[:, 1]
        nn_index = indices[:, 1]

        # Distancias en dimensión m+1
        dist_m1 = np.linalg.norm(Y_m1 - Y_m1[nn_index], axis=1)

        # Condiciones para falsos vecinos
        condition1 = np.abs(dist_m1 - R_m) / R_m > Rtol
        condition2 = dist_m1 > Atol

        FNN = np.sum(condition1 | condition2)
        perc_FNN.append(100 * FNN / M)

    return perc_FNN

# Calcular FNN con el tau óptimo
perc_FNN = compute_FNN_curve(ts, tau_optimo, max_dim=max_dim)

plt.figure(figsize=(8, 6))
plt.plot(range(1, len(perc_FNN) + 1), perc_FNN, marker='o', color='b', linewidth=1.5)
plt.xlabel("Dimensión m", fontsize=12)
plt.ylabel("Porcentaje FNN", fontsize=12)
plt.title("False Nearest Neighbors (FNN)", fontsize=13)
plt.xlim(0, max_dim)
plt.xticks(np.arange(0, max_dim + 1, 1))
plt.yticks(np.arange(0, 101, 10))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Encontrar dimensión óptima (primer valor donde FNN < 10%)
m_optimo = 2
for i, val in enumerate(perc_FNN):
    if val < 10:
        m_optimo = i + 1
        break

print(f"\nDimensión m óptima sugerida por FNN: {m_optimo}")

# ======================================================
# 10. RESUMEN DEL ANÁLISIS
# ======================================================

print("\n" + "="*50)
print("RESUMEN DEL ANÁLISIS")
print("="*50)
print(f"Longitud de la serie: {len(ts)} puntos")
print(f"Tiempo total: {T} unidades")
print(f"Resolución temporal: {dt}")
print(f"Parámetro ρ: {rho}")
print(f"\nParámetros de embedding sugeridos:")
print(f"  - Retardo óptimo (τ): {tau_optimo} (primer mínimo información mutua)")
print(f"  - Dimensión óptima (m): {m_optimo} (FNN < 10%)")
print(f"\nParámetros usados originalmente:")
print(f"  - Dimensión (m): {m}")
print(f"  - Retardo (τ): {tau}")

# Reacción BZ

Este notebook analiza la serie temporal del sistema Belousov-Zhabotinsky (b=3.46) para optimizar el embedding de Takens. Calcula autocorrelación e información mutua para determinar el retardo óptimo (τ), y False Nearest Neighbors (FNN) para la dimensión óptima (m). Incluye visualización de la serie temporal y proyecciones del embedding.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from sklearn.neighbors import NearestNeighbors
from gtda.time_series import SingleTakensEmbedding

# ======================================================
# 1. SISTEMA DE BELOUSOV-ZHABOTINSKY (BZ)
# ======================================================

def bz_system(t, X, a, b):
    """Ecuaciones diferenciales del sistema BZ."""
    x, y = X
    dx = a - x - (4*x*y)/(1 + x**2)
    dy = b*x*(1 - y/(1 + x**2))
    return [dx, dy]

# ======================================================
# 2. GENERAR SERIE TEMPORAL x(t)
# ======================================================

def generate_bz_ts(b, dt=0.01, T=200):
    """Genera la serie temporal x(t) del sistema BZ."""
    t_span = (0, T)
    t_eval = np.arange(0, T, dt)
    a = 10.0
    x0 = [2, 7]  # Condiciones iniciales

    sol = solve_ivp(
        bz_system,
        t_span,
        x0,
        t_eval=t_eval,
        args=(a, b),
        rtol=1e-11,
        atol=1e-12
    )
    return sol.y[0]  # Retorna x(t)

# ======================================================
# 3. CONFIGURACIÓN DE PARÁMETROS
# ======================================================

b = 3.499        # Cerca de la bifurcación de Hopf
dt = 0.01       # Paso de tiempo
T = 100         # Tiempo total de simulación

# Parámetros de embedding (se optimizarán después)
tau = 59        # Retardo temporal inicial
m = 2           # Dimensión de embedding inicial

# ======================================================
# 4. GENERAR Y GRAFICAR LA SERIE TEMPORAL
# ======================================================

ts = generate_bz_ts(b, dt, T)
time = np.arange(0, T, dt)

fig = plt.figure(figsize=(15, 10))

# Subplot 1: Serie temporal completa
ax1 = plt.subplot(2, 2, 1)
ax1.plot(time, ts, 'b-', linewidth=0.8, alpha=0.8)
ax1.set_xlabel('Tiempo (t)', fontsize=12)
ax1.set_ylabel('x(t)', fontsize=12)
ax1.set_title(f'Serie Temporal del Sistema BZ (b={b})', fontsize=14)
ax1.margins(x=0.0)
ax1.grid(True, alpha=0.3)

# Subplot 2: Zoom de la serie temporal
ax2 = plt.subplot(2, 2, 2)
zoom_start, zoom_end = 0, T
zoom_mask = (time >= zoom_start) & (time <= zoom_end)
ax2.plot(time[zoom_mask], ts[zoom_mask], 'r-', linewidth=0.8)
ax2.set_xlabel('Tiempo (t)', fontsize=12)
ax2.set_ylabel('x(t)', fontsize=12)
ax2.set_title(f'Zoom: t ∈ [{zoom_start}, {zoom_end}]', fontsize=14)
ax2.margins(x=0.0)
ax2.grid(True, alpha=0.3)

# ======================================================
# 5. CREAR EMBEDDING DE TAKENS
# ======================================================

embedder = SingleTakensEmbedding(
    parameters_type="fixed",
    n_jobs=1,
    time_delay=tau,
    dimension=m,
    stride=2
)

X = embedder.fit_transform(ts[zoom_mask].reshape(-1, 1))
N = X.shape[0]

print(f"Dimensiones del embedding: {X.shape}")
print(f"Parámetros: τ={tau}, m={m}")
print(f"Número de puntos en el espacio embebido: {N}")

# ======================================================
# 6. PROYECCIONES 2D DEL EMBEDDING
# ======================================================

fig2, axes = plt.subplots(1, 2, figsize=(12, 5))

# Proyección XY
scatter1 = axes[0].scatter(X[:, 0], X[:, 1], c=np.arange(N), cmap='viridis', s=5, alpha=0.7)
axes[0].set_xlabel('x(t)')
axes[0].set_ylabel(f'x(t+{tau})')
axes[0].set_title('Proyección XY')
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=axes[0], label='Tiempo')

# Proyección XZ (si m>=3)
if m >= 3:
    scatter2 = axes[1].scatter(X[:, 0], X[:, 2], c=np.arange(N), cmap='viridis', s=5, alpha=0.7)
    axes[1].set_xlabel('x(t)')
    axes[1].set_ylabel(f'x(t+{2*tau})')
    axes[1].set_title('Proyección XZ')
    axes[1].grid(True, alpha=0.3)
    plt.colorbar(scatter2, ax=axes[1], label='Tiempo')

plt.tight_layout()
plt.show()

# ======================================================
# 7. ANÁLISIS DE AUTOCORRELACIÓN
# ======================================================

fig3, axes = plt.subplots(1, 2, figsize=(12, 5))

from scipy.signal import correlate

autocorr = correlate(ts - np.mean(ts), ts - np.mean(ts), mode='full')
autocorr = autocorr[len(autocorr)//2:]
autocorr = autocorr / autocorr[0]

# Primer cruce por cero
zero_crossings = np.where(np.diff(np.sign(autocorr)))[0]
first_zero_crossing = zero_crossings[0] if len(zero_crossings) > 0 else len(autocorr) - 1

axes[0].plot(autocorr[:200], 'b-', linewidth=1.5)
axes[0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[0].axvline(x=first_zero_crossing, color='g', linestyle='--',
                alpha=0.7, label=f'Primer cruce: {first_zero_crossing}')
axes[0].set_xlabel('Lag (k)')
axes[0].set_ylabel('Autocorrelación')
axes[0].set_title('Autocorrelación de x(t)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ======================================================
# 8. INFORMACIÓN MUTUA
# ======================================================

max_lag = 200

def mutual_information_estimate(series, max_lag=max_lag):
    """Calcula la información mutua para diferentes lags."""
    n_bins = 20
    mi_vals = []

    for lag in range(1, max_lag + 1):
        shifted = series[lag:]
        original = series[:-lag]

        hist, _, _ = np.histogram2d(original, shifted, bins=n_bins)
        hist = hist / np.sum(hist)

        mi = 0
        for i in range(n_bins):
            for j in range(n_bins):
                if hist[i, j] > 0:
                    p_x = np.sum(hist[i, :])
                    p_y = np.sum(hist[:, j])
                    mi += hist[i, j] * np.log(hist[i, j] / (p_x * p_y))
        mi_vals.append(mi)

    return np.array(mi_vals)

mi_vals = mutual_information_estimate(ts, max_lag=max_lag)
first_min = np.argmin(mi_vals) + 1

axes[1].plot(range(1, max_lag + 1), mi_vals, 'b-', linewidth=1.5)
axes[1].axvline(x=first_min, color='g', linestyle='--',
                alpha=0.7, label=f'Primer mínimo: {first_min}')
axes[1].set_xlabel('Lag (τ)')
axes[1].set_ylabel('Información Mutua')
axes[1].set_title('Información Mutua vs Lag')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Actualizar tau con el valor óptimo
tau_optimo = first_min
print(f"\nτ óptimo sugerido por información mutua: {tau_optimo}")

# ======================================================
# 9. FNN (FALSE NEAREST NEIGHBORS)
# ======================================================

max_dim = 12

def compute_FNN_curve(ts, tau, max_dim=max_dim, Rtol=15, Atol=2):
    """Calcula el porcentaje de falsos vecinos cercanos (FNN)."""
    N = len(ts)
    perc_FNN = []

    for m in range(1, max_dim + 1):
        M = N - (m * tau)
        if M <= 0:
            break

        # Vectores en dimensión m
        Y_m = np.zeros((M, m))
        for i in range(m):
            Y_m[:, i] = ts[i * tau : i * tau + M]

        # Vectores en dimensión m+1
        Y_m1 = np.zeros((M, m + 1))
        for i in range(m + 1):
            Y_m1[:, i] = ts[i * tau : i * tau + M]

        # Vecinos más cercanos
        nbrs = NearestNeighbors(n_neighbors=2).fit(Y_m)
        distances, indices = nbrs.kneighbors(Y_m)

        R_m = distances[:, 1]
        nn_index = indices[:, 1]

        # Distancias en dimensión m+1
        dist_m1 = np.linalg.norm(Y_m1 - Y_m1[nn_index], axis=1)

        # Condiciones para falsos vecinos
        condition1 = np.abs(dist_m1 - R_m) / R_m > Rtol
        condition2 = dist_m1 > Atol

        FNN = np.sum(condition1 | condition2)
        perc_FNN.append(100 * FNN / M)

    return perc_FNN

# Calcular FNN con el tau óptimo
perc_FNN = compute_FNN_curve(ts, tau_optimo, max_dim=max_dim)

plt.figure(figsize=(8, 6))
plt.plot(range(1, len(perc_FNN) + 1), perc_FNN, marker='o', color='b', linewidth=1.5)
plt.xlabel("Dimensión m", fontsize=12)
plt.ylabel("Porcentaje FNN", fontsize=12)
plt.title("False Nearest Neighbors (FNN)", fontsize=13)
plt.xlim(0, max_dim)
plt.xticks(np.arange(0, max_dim + 1, 1))
plt.yticks(np.arange(0, 101, 10))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Encontrar dimensión óptima (primer valor donde FNN < 10%)
m_optimo = 2
for i, val in enumerate(perc_FNN):
    if val < 10:
        m_optimo = i + 1
        break

print(f"\nDimensión m óptima sugerida por FNN: {m_optimo}")

# ======================================================
# 10. RESUMEN DEL ANÁLISIS
# ======================================================

print("\n" + "="*50)
print("RESUMEN DEL ANÁLISIS")
print("="*50)
print(f"Longitud de la serie: {len(ts)} puntos")
print(f"Tiempo total: {T} unidades")
print(f"Resolución temporal: {dt}")
print(f"Parámetro b: {b}")
print(f"\nParámetros de embedding sugeridos:")
print(f"  - Retardo óptimo (τ): {tau_optimo} (primer mínimo información mutua)")
print(f"  - Dimensión óptima (m): {m_optimo} (FNN < 10%)")
print(f"\nParámetros usados originalmente:")
print(f"  - Dimensión (m): {m}")
print(f"  - Retardo (τ): {tau}")